<h1><strong>Intelligent Fuel Consumption Prediction<strong/><h1>
<div style="display: flex;">
<img  src="imges/fuel.webp" width="200" height="150">
<img  src="imges/vassel.webp" width="200" height="150">
<div/>

<h4>Importing the Dependencies<h4/>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import  mean_absolute_error
from sklearn.model_selection import KFold
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score



In [ ]:
df = pd.read_csv('data/BRAVERUS_20150101_TO_20181231.csv')
df_cpy=df.copy()

In [ ]:
df_cpy.head()

In [ ]:
df_cpy.describe()

In [ ]:
columns_to_drop = [col for col in df_cpy.columns if df_cpy[col].nunique() == 1]
df_cpy.drop(columns=columns_to_drop, axis=1,inplace=True)

In [ ]:
df_cpy.info()

In [ ]:
df_cpy['DATED'] = pd.to_datetime(df_cpy['DATED'], format="%d/%m/%Y") # DATED -> object 
df_cpy['OPERATION'] = df_cpy['OPERATION'].fillna(df_cpy['OPERATION'].mode()[0])

In [ ]:
OPERATION_encoder = LabelEncoder()
df_cpy['OPERATION']=OPERATION_encoder.fit_transform(df_cpy['OPERATION'])

In [ ]:
VOYAGE_encoder = LabelEncoder()
df_cpy['VOYAGE']=VOYAGE_encoder.fit_transform(df_cpy['VOYAGE'])

In [ ]:
TO_encoder = LabelEncoder()
df_cpy['TO']=TO_encoder.fit_transform(df_cpy['TO'])

In [ ]:
fuel_types = [col for col in df_cpy.columns if col.endswith('CONS')]
df_cpy['Total Fuel Consumption'] = df_cpy[fuel_types].sum(axis=1)
df_cpy.drop(columns=fuel_types,inplace=True)

In [ ]:
df_cpy.head()

In [ ]:
numeric_columns = df_cpy.select_dtypes(include=['float64', 'int64','int32']).columns

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(15, 20), constrained_layout=True)

for i, col in enumerate([col for col in numeric_columns if col not in {'OPERATION','VOYAGE','TO'}]):
    row = i // 4
    col_num = i % 4
    ax = axes[row, col_num] if len(numeric_columns) > 1 else axes[col_num]
    
    sns.boxplot(y=df_cpy[col], ax=ax)
    ax.set_title(f'{col} BOXPLOT')
    ax.set_ylim(bottom=0, top=df_cpy[col].max() * 1.10)  

In [ ]:
df_cpy.describe()

In [ ]:
"""before handaling Outliers:
     r2_score  mean_absolute_error  mean_squared_error
rfr  0.965804             1.638191            7.795145
gbr  0.972402             1.638914            6.291215
xgb  0.961786             1.632974            8.711122
"""
columns_with_outliers = ['RPM', 'SLIP', 'SPEED', 'WIND', 'CUR SPD', 'STW', 'SWELL']

for col in columns_with_outliers:
    Q1 = df_cpy[col].quantile(0.25)
    Q3 = df_cpy[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_cpy=df_cpy[(df_cpy[col] >= lower_bound) & (df_cpy[col] <= upper_bound)]


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(15, 20), constrained_layout=True)

for i, col in enumerate([col for col in numeric_columns if col not in {'OPERATION','VOYAGE','TO'}]):
    row = i // 4
    col_num = i % 4
    ax = axes[row, col_num] if len(numeric_columns) > 1 else axes[col_num]
    
    sns.boxplot(y=df_cpy[col], ax=ax)
    ax.set_title(f'{col} BOXPLOT')
    ax.set_ylim(bottom=0, top=df_cpy[col].max() * 1.10)  

In [ ]:
df_cpy.set_index('DATED')['Total Fuel Consumption'].plot(figsize=(20,6),title="Total Fuel Consumption Over Time")

In [ ]:
decoded_labels = OPERATION_encoder.inverse_transform(df_cpy['OPERATION'])
operation_counts = pd.Series(decoded_labels).value_counts()

plt.figure(figsize=(8,8))
plt.pie(operation_counts, labels=operation_counts.index, autopct='%1.1f%%')
plt.title('Distribution of Operations')
plt.show()

In [ ]:

corr_matrix = df_cpy[numeric_columns].drop(columns=['OPERATION','VOYAGE','TO']).corr()
plt.figure(figsize=(15, 12))  
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', annot_kws={"size": 10}, linewidths=0.5)
plt.title('Heatmap of Numeric Columns Correlation', fontsize=16)
plt.show()


In [ ]:
corr_matrix

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(15, 20), constrained_layout=True)

for i, col in enumerate([col for col in numeric_columns if col not in {'OPERATION','VOYAGE','TO'}]):
    row = i // 4
    col_num = i % 4
    ax = axes[row, col_num]

    sns.histplot(df_cpy[col], bins=50, kde=True, color='blue', ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(f'{col}')
    ax.set_ylabel('Count')
plt.show()

In [ ]:
sns.pairplot(df_cpy[numeric_columns].drop(columns={'OPERATION','VOYAGE','TO'}))
plt.show()

Train the models

In [ ]:

def perform_cross_validation(model, X, y, cv):
    kfold = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kfold, scoring='r2')
    return scores

def train_model(X_train, y_train, n_estimators, model):
    
    if model == 'rfr':
        model = RandomForestRegressor(n_estimators=n_estimators)
    elif model == 'gbr':
        model = GradientBoostingRegressor(n_estimators=n_estimators)
    elif model == 'xgb':
        model = xgb.XGBRegressor(n_estimators=n_estimators)
    else:
        return None, None, None

    model.fit(X_train, y_train)
    
    return model

In [ ]:
target='Total Fuel Consumption'

y=df_cpy[target]
X=df_cpy[numeric_columns].drop(columns=[target,'SPEED','STEAM','SWELL','CARGOCARRIED','SLIP','CUR SPD','VOYAGE'])
X.columns


In [ ]:
X_scaled = minmax_scale(X, feature_range=(0, 1))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
rfr_model = train_model(X_train,y_train, n_estimators=100, model='rfr')
gbr_model = train_model(X_train,y_train, n_estimators=100,model='gbr')
xgb_model = train_model(X_train,y_train, n_estimators=100, model='xgb')

accuracy check

In [ ]:
models=[rfr_model,gbr_model,xgb_model]
training_scores=[]
testing_scores=[]
individual_scores=[]
mae_scores=[]
mse_scores=[]

In [65]:
for model in models:
    y_pred=model.predict(X_test)  
    training_scores.append(perform_cross_validation(model,X_train,y_train,20).mean())
    testing_scores.append(perform_cross_validation(model,X_test,y_test,20).mean())
    individual_scores.append(r2_score(y_test,y_pred))
    mae_scores.append(mean_absolute_error( y_test,y_pred))
    mse_scores.append(mean_squared_error(y_test,y_pred))

KeyboardInterrupt: 

In [ ]:


index = ["rfr", "gbr","xgb"]
compare_table = pd.DataFrame({
    'r2_training_scores': training_scores,
    'r2_testing_scores': testing_scores,
    'r2_individual_scores': individual_scores,
    'mean_absolute_error': mae_scores,
    'mean_squared_error': mse_scores
}, index=index)

compare_table